# ADNI Diagnosis Classification — Training & Saving Pipeline
## LightGBM + Soft-Vote Ensemble | Full Stream & Algerian Stream

This notebook:
1. Loads and preprocesses the ADNI dataset (identical pipeline from experimenting notebooks)
2. Trains **LightGBM** and a **soft-vote ensemble** (top-3 neural models) for both Full and Algerian streams
3. **Saves** all artifacts (models, scalers, feature lists, label encoders) for local inference
4. Provides **`predict_instance()`** helpers for single-visit inference on new data


## 0. Imports & Configuration

In [1]:
import warnings, os, random, time
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("LightGBM not installed — run: pip install lightgbm")

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

CLASS_NAMES = ['CN', 'MCI', 'Dementia']
N_CLASSES   = 3

# ── Training budget ──────────────────────────────────────────────
BATCH_SIZE   = 1024
FULL_EPOCHS  = 80
FULL_PATIENCE= 12
MIN_F1       = 0.50

# ── Output directory for saved models ────────────────────────────
SAVE_ROOT = './diagnosis_models'
os.makedirs(f'{SAVE_ROOT}/full', exist_ok=True)
os.makedirs(f'{SAVE_ROOT}/alg',  exist_ok=True)
print(f'Models will be saved to: {SAVE_ROOT}/')


Device: cpu
Models will be saved to: ./diagnosis_models/


## 1. Data Paths — Update These

In [2]:
# ── UPDATE THESE PATHS ───────────────────────────────────────────
CSV_PATH = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/adni_stability.csv'
MTA_PATH = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/MTA_labels_final.csv'
AMY_PATH = '/kaggle/input/datasets/baraafzlalagui/adni-data-selected/amy_dataset.csv'


## 2. Feature Definitions

In [3]:
TARGET   = 'DIAGNOSIS_LABEL'
ID_COLS  = ['RID', 'VISDATE', 'VISCODE2']
DROP_BASE = set(ID_COLS + [TARGET, 'DXMDUE','DXDSEV','DXDEP','DXCONFID'])

COLS_LIST_ALG = [
    "RID", "VISCODE2", "PTGENDER", "age", "PTHAND", "PTMARRY", "PTEDUCAT", "PTWORK", "PTNOTRT",
    "VISDATE", "MMDATE", "MMYEAR", "MMMONTH", "MMDAY", "MMREAD", "MMWRITE", "MMDRAW", "MMREPEAT",
    "MMSEASON", "MMHOSPIT", "MMFLOOR", "MMCITY", "WORD1", "WORD2", "WORD3", "MMSCORE", "MOCA", "CUBE",
    "CLOCKCON", "CLOCKNO", "CLOCKHAN", "DIGFOR", "DIGBACK", "SERIAL1", "SERIAL2", "SERIAL3", "SERIAL4",
    "SERIAL5", "REPEAT1", "REPEAT2", "FFLUENCY", "FAQFORM", "FAQFINAN", "FAQSHOP", "FAQGAME", "FAQBEVG",
    "FAQMEAL", "FAQEVENT", "FAQTV", "FAQREM", "FAQTRAVL", "FAQ", "Creatinine", "Calcium", "Direct Bilirubin",
    "Platelet Ct.", "Red Blood Cell Count", "Thyroid-stimulating hormone", "Total Bilirubin", "glucose",
    "Vitamin B12", "White Blood Cell Count", "Abeta42", "Abeta40", "Abeta_ratio", "Homocysteine", "Hemoglobin A1C",
    "Tau181", "pTau181", "Gamma-Glutamyltransferase", "Hematocrit", "Hemoglobin",
    "MOTHDEM", "MOTHAD", "MOTHSXAGE", "FATHDEM", "FATHAD", "FATHSXAGE", "SIBGENDER", "SIBDEMENT", "SIBAD",
    "SIBSXAGE", "IHSYMPTOM", "IHDESC", "IHCHRON", "IHSEVER", "IHPRESENT", "IHSURG", "MH19OTHR", "MHPSYCH",
    "MH2NEURL", "MH3HEAD", "MH4CARD", "MH5RESP", "MH6HEPAT", "MH7DERM", "MH8MUSCL", "MH9ENDO", "MH14BALCH",
    "MH14CALCH", "MH10GAST", "MH11HEMA", "MH12RENA", "MH13ALLE", "MH14ALCH", "MH14AALCH", "MH17MALI",
    "MH18SURG", "MH15DRUG", "MH15ADRUG", "MH15BDRUG", "MH16SMOK", "MH16ASMOK", "MH16BSMOK", "MH16CSMOK",
    "BSXSYMNO", "BSXSEVER", "BSXCHRON", "KEYMED", "CMMED", "CMDOSE", "CMREASON",
    "DIAGNOSIS_LABEL", 'AMYLOID_STATUS', 'MTA_ATROPHY'
]


## 3. Data Loading

In [4]:
raw = pd.read_csv(CSV_PATH, low_memory=False)
raw.drop(columns=['Unnamed: 0', 'DIAGNOSIS', 'TRAJECTORY_CLEANED', 'OBS_SPAN_YEARS', 'STABILITY_LABEL'],
         inplace=True, errors='ignore')

# Encode DIAGNOSIS_LABEL
raw = raw[raw['DIAGNOSIS_LABEL'].notna()].reset_index(drop=True)
le_global = LabelEncoder()
raw['DIAGNOSIS_LABEL'] = le_global.fit_transform(raw['DIAGNOSIS_LABEL'].astype(str))
print(f"Classes: {le_global.classes_}  →  {le_global.transform(le_global.classes_)}")
assert len(le_global.classes_) == 3

# Merge MTA & Amyloid
mta = pd.read_csv(MTA_PATH)
amy = pd.read_csv(AMY_PATH)
raw = raw.merge(mta[['RID','VISCODE2','MTA_ATROPHY']].drop_duplicates(['RID','VISCODE2']),
                on=['RID','VISCODE2'], how='left')
raw = raw.merge(amy[['RID','VISCODE2','AMYLOID_STATUS']].drop_duplicates(['RID','VISCODE2']),
                on=['RID','VISCODE2'], how='left')

print(f"Dataset shape: {raw.shape}")
print(f"DIAGNOSIS_LABEL distribution:\n{pd.Series(raw['DIAGNOSIS_LABEL']).value_counts()}")

FEATURE_COLS_FULL = [c for c in raw.columns if c not in DROP_BASE]
FEATURE_COLS_ALG  = [c for c in COLS_LIST_ALG if c in raw.columns and c not in DROP_BASE]
print(f'Stream FULL: {len(FEATURE_COLS_FULL)} features')
print(f'Stream ALG : {len(FEATURE_COLS_ALG)} features')


Classes: ['CN' 'Dementia' 'MCI']  →  [0 1 2]
Dataset shape: (20696, 144)
DIAGNOSIS_LABEL distribution:
DIAGNOSIS_LABEL
2    8355
0    8215
1    4126
Name: count, dtype: int64
Stream FULL: 136 features
Stream ALG : 121 features


## 4. Domain Group Builder

In [5]:
def build_group_map(feature_cols):
    cog = [c for c in feature_cols if any(x in c for x in
        ['MMDATE','MMYEAR','MMMONTH','MMDAY','MMREAD','MMWRITE','MMDRAW','MMREPEAT',
         'MMSEASON','MMHOSPIT','MMFLOOR','MMCITY','WORD','MMSCORE','CUBE','CLOCK',
         'DIGFOR','DIGBACK','SERIAL','REPEAT','FFLUENCY','MOCA'])]
    fnc = [c for c in feature_cols if 'FAQ' in c]
    bio = [c for c in feature_cols if any(x in c for x in
        ['Abeta','Tau','pTau','Alkaline','Calcium','Cholesterol','Creatinine','Glucose',
         'HDL','Hemoglobin','LDL','Thyroid','Protein','Triglycerides','Vitamin',
         'Homocysteine','Cystatin','Bilirubin','Urea','Phosphatase','Transferase',
         'Hematocrit','Methylmalonic','Lactate','Platelet','Red Blood','White Blood',
         'eGFR','glucose','protein','Alanine','Aspartate','Gamma-Glutamyl',
         'High Sensitivity', 'AMYLOID_STATUS', 'MTA_ATROPHY'])]
    dem = [c for c in feature_cols if any(x in c for x in
        ['PTGENDER','age','PTHAND','PTMARRY','PTEDUCAT','PTWORK','PTNOTRT','GENOTYPE',
         'MOTHDEM','MOTHAD','FATHDEM','FATHAD','SIBGENDER','SIBDEMENT','SIBAD'])]
    mh  = [c for c in feature_cols if c.startswith('MH') or
           any(x in c for x in ['IHSYMPTOM','IHDESC','IHCHRON','IHSEVER','IHPRESENT',
                                 'IHSURG','KEYMED','BSXSYMNO','CMMED','CMDOSE','CMREASON','CMREASONO'])]
    assigned = set(cog + fnc + bio + dem + mh)
    other    = [c for c in feature_cols if c not in assigned]
    return {'cognitive': cog, 'functional': fnc, 'biomarker': bio,
            'demographic': dem, 'medical_history': mh, 'other': other}

def build_idx(all_feat, group_map):
    idx = {}
    for g, cols in group_map.items():
        cs = set(cols)
        idx[g] = [i for i, c in enumerate(all_feat) if c in cs]
    for c in all_feat:
        if c.endswith('_missing'):
            base = c[:-len('_missing')]
            for g, cols in group_map.items():
                if base in cols:
                    if c in all_feat:
                        idx[g].append(all_feat.index(c))
                    break
    return {g: list(set(v)) for g, v in idx.items() if v}

GM_FULL  = build_group_map(FEATURE_COLS_FULL)
GM_ALG   = build_group_map(FEATURE_COLS_ALG)
print("Domain groups built.")


Domain groups built.


## 5. Preprocessing Pipeline

In [6]:
def preprocess_stream(raw_df, feature_cols, target='DIAGNOSIS_LABEL', rid='RID'):
    """
    Patient-median imputation → missing flags → label-encode categoricals.
    Returns X, y, all_feat_list, label_encoder, pat_median_dict, global_median_dict, str_le_map.
    The extra dicts are saved for inference on new data.
    """
    avail = [c for c in feature_cols if c in raw_df.columns]
    df    = raw_df[[rid] + avail + [target]].copy()

    num_cols = df[avail].select_dtypes(include=[np.number]).columns.tolist()
    str_cols = [c for c in avail if c not in num_cols]

    # Missing flags
    miss_cols = []
    for col in num_cols:
        if df[col].isnull().mean() > 0.05:
            df[f'{col}_missing'] = df[col].isnull().astype(float)
            miss_cols.append(f'{col}_missing')

    # Per-patient median → global fallback
    pat_med = df.groupby(rid)[num_cols].median()
    glb_med = df[num_cols].median().fillna(0)

    def impute(group):
        for col in num_cols:
            pm = pat_med.loc[group.name, col] if group.name in pat_med.index else np.nan
            group[col] = group[col].fillna(glb_med[col] if pd.isna(pm) else pm)
        return group

    df = df.groupby(rid, group_keys=False).apply(impute)

    # Label-encode string cols, store encoders for inference
    str_le_map = {}
    for col in str_cols:
        df[col] = df[col].fillna('not_recorded')
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        str_le_map[col] = le

    df.drop(columns=[rid], inplace=True)
    all_feat = avail + miss_cols
    labeled  = df[df[target].notna()].reset_index(drop=True)
    X = labeled[all_feat].values
    le_lbl = LabelEncoder()
    y = le_lbl.fit_transform(labeled[target].values.astype(int))
    return X, y, all_feat, le_lbl, pat_med, glb_med, str_le_map

print('Preprocessing FULL stream...')
X_full, y_full, AF_FULL, le_full, pat_med_full, glb_med_full, str_le_full = \
    preprocess_stream(raw, FEATURE_COLS_FULL)
print(f'  X={X_full.shape}')

print('Preprocessing ALG stream...')
X_alg, y_alg, AF_ALG, le_alg, pat_med_alg, glb_med_alg, str_le_alg = \
    preprocess_stream(raw, FEATURE_COLS_ALG)
print(f'  X={X_alg.shape}')

IDX_FULL = build_idx(AF_FULL, GM_FULL)
IDX_ALG  = build_idx(AF_ALG,  GM_ALG)
IDX_FULL = {g: v for g, v in IDX_FULL.items() if v}
IDX_ALG  = {g: v for g, v in IDX_ALG.items()  if v}


Preprocessing FULL stream...
  X=(20696, 252)
Preprocessing ALG stream...
  X=(20696, 222)


In [7]:
def split_scale(X, y):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    sc  = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr).astype(np.float32)
    Xte_s = sc.transform(Xte).astype(np.float32)
    cw    = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return Xtr_s, Xte_s, ytr, yte, sc, torch.tensor(cw, dtype=torch.float32).to(device)

X_tr_full, X_te_full, y_tr_full, y_te_full, scaler_full, cw_full = split_scale(X_full, y_full)
X_tr_alg,  X_te_alg,  y_tr_alg,  y_te_alg,  scaler_alg,  cw_alg  = split_scale(X_alg,  y_alg)

Xtr_f, Xval_f, ytr_f, yval_f = train_test_split(X_tr_full, y_tr_full, test_size=0.15,
                                                   stratify=y_tr_full, random_state=SEED)
Xtr_a, Xval_a, ytr_a, yval_a = train_test_split(X_tr_alg,  y_tr_alg,  test_size=0.15,
                                                   stratify=y_tr_alg,  random_state=SEED)
N_FEAT_FULL = X_tr_full.shape[1]
N_FEAT_ALG  = X_tr_alg.shape[1]
print(f'FULL → Train={X_tr_full.shape[0]:,}  Test={X_te_full.shape[0]:,}  Feat={N_FEAT_FULL}')
print(f'ALG  → Train={X_tr_alg.shape[0]:,}   Test={X_te_alg.shape[0]:,}   Feat={N_FEAT_ALG}')


FULL → Train=16,556  Test=4,140  Feat=252
ALG  → Train=16,556   Test=4,140   Feat=222


## 6. Training Utilities (FocalLoss, SupCon, OneCycleLR)

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma
        self.ls     = label_smoothing

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.weight,
                               label_smoothing=self.ls, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class SupConLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.T = temperature

    def forward(self, features, labels):
        # features: (B, D) L2-normalised
        features = F.normalize(features, dim=1)
        sim  = torch.matmul(features, features.T) / self.T
        B    = sim.shape[0]
        mask = torch.eye(B, dtype=torch.bool, device=sim.device)
        sim  = sim.masked_fill(mask, float('-inf'))
        pos  = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
        pos  = pos.masked_fill(mask, 0)
        if pos.sum() == 0:
            return torch.tensor(0., device=sim.device)
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        loss = -(log_prob * pos).sum(1) / pos.sum(1).clamp(min=1)
        return loss.mean()


def make_loader(X, y, bs=BATCH_SIZE, shuffle=True):
    return DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32),
                                    torch.tensor(y, dtype=torch.long)),
                      batch_size=bs, shuffle=shuffle)


def fit(model, X_tr, y_tr, X_val, y_val,
        lr=3e-4, sc_alpha=0.3, sc_temp=0.1, focal_gamma=2.0,
        class_weights_tensor=None):
    """Train with joint SupCon+FocalLoss, OneCycleLR, early stopping on val F1."""
    tr_loader  = make_loader(X_tr, y_tr)
    val_loader = make_loader(X_val, y_val, shuffle=False)
    opt   = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(tr_loader),
                       epochs=FULL_EPOCHS, pct_start=0.15)
    fl    = FocalLoss(weight=class_weights_tensor, gamma=focal_gamma)
    scl   = SupConLoss(temperature=sc_temp)

    best_f1, best_state, patience_cnt = 0., None, 0
    history = []
    for epoch in range(FULL_EPOCHS):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            if hasattr(model, 'project'):
                logits = model(Xb)
                proj   = model.project(Xb)
                loss   = sc_alpha * scl(proj, yb) + (1 - sc_alpha) * fl(logits, yb)
            else:
                logits = model(Xb)
                loss   = fl(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()

        # Validation
        model.eval()
        preds = []
        with torch.no_grad():
            for Xb, _ in val_loader:
                preds.append(model(Xb.to(device)).argmax(1).cpu())
        preds = torch.cat(preds).numpy()
        vf1 = f1_score(y_val, preds, average='macro', zero_division=0)
        history.append(vf1)

        if vf1 > best_f1:
            best_f1, best_state, patience_cnt = vf1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_cnt += 1
        if patience_cnt >= FULL_PATIENCE:
            break

    if best_state:
        model.load_state_dict(best_state)
    return best_f1, history


@torch.no_grad()
def get_probs(model, X):
    model.eval()
    loader = make_loader(X, np.zeros(len(X)), shuffle=False)
    probs  = []
    for Xb, _ in loader:
        probs.append(F.softmax(model(Xb.to(device)), dim=1).cpu().numpy())
    return np.vstack(probs)


def score_model(model, X_te, y_te, name, stream_tag):
    probs = get_probs(model, X_te)
    preds = probs.argmax(1)
    f1    = f1_score(y_te, preds, average='macro', zero_division=0)
    acc   = accuracy_score(y_te, preds)
    try:
        auc = roc_auc_score(np.eye(N_CLASSES)[y_te], probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    print(f'  [{stream_tag}] {name}: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    print(classification_report(y_te, preds, target_names=CLASS_NAMES, zero_division=0))
    return {'name': name, 'stream': stream_tag, 'accuracy': acc, 'f1_macro': f1, 'auc_ovr': auc}

print("Training utilities ready.")


Training utilities ready.


## 7. Neural Network Architectures (GRNNet, DeepResNet-SE, CompactDomainFTT, DualStreamDomainNet)

In [9]:
# ── GRNNet ──────────────────────────────────────────────────────
class GRNBlock(nn.Module):
    def __init__(self, in_d, hidden_d, out_d, dropout=0.2):
        super().__init__()
        self.fc1  = nn.Linear(in_d, hidden_d); self.fc2 = nn.Linear(hidden_d, out_d)
        self.fc3  = nn.Linear(in_d, out_d, bias=False)
        self.gate = nn.Linear(in_d, out_d)
        self.bn   = nn.BatchNorm1d(out_d); self.drop = nn.Dropout(dropout)
        self.skip = nn.Linear(in_d, out_d, bias=False) if in_d != out_d else nn.Identity()
    def forward(self, x):
        h = F.elu(self.fc1(x)); h = self.drop(h)
        h = self.fc2(h) + self.fc3(x)
        g = torch.sigmoid(self.gate(x))
        return self.bn(g * h + (1 - g) * self.skip(x))

class GRNNet(nn.Module):
    def __init__(self, n_features, hidden=256, n_blocks=5, dropout=0.25, proj_dim=64, n_out=3):
        super().__init__()
        self.entry     = nn.Sequential(nn.Linear(n_features, hidden), nn.LayerNorm(hidden), nn.GELU())
        self.blocks    = nn.ModuleList([GRNBlock(hidden, hidden*2, hidden, max(0.05, dropout-i*.03)) for i in range(n_blocks)])
        self.projector = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, proj_dim))
        self.head      = nn.Sequential(nn.Linear(hidden, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64, n_out))
    def _encode(self, x):
        x = self.entry(x)
        for b in self.blocks: x = b(x)
        return x
    def project(self, x): return self.projector(self._encode(x))
    def forward(self, x): return self.head(self._encode(x))

# ── DeepResNet-SE ────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, dim, reduction=4):
        super().__init__()
        self.se = nn.Sequential(nn.Linear(dim, dim//reduction), nn.ReLU(),
                                 nn.Linear(dim//reduction, dim), nn.Sigmoid())
    def forward(self, x): return x * self.se(x)

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim*2), nn.BatchNorm1d(dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim*2, dim), nn.BatchNorm1d(dim))
        self.se  = SEBlock(dim); self.act = nn.GELU()
    def forward(self, x): return self.act(x + self.se(self.block(x)))

class DeepResNet(nn.Module):
    def __init__(self, n_features, hidden=256, n_blocks=6, dropout=0.25, proj_dim=64, n_out=3):
        super().__init__()
        self.entry     = nn.Sequential(nn.Linear(n_features, hidden), nn.BatchNorm1d(hidden), nn.GELU())
        self.blocks    = nn.ModuleList([ResBlock(hidden, max(0.05, dropout-i*.025)) for i in range(n_blocks)])
        self.projector = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, proj_dim))
        self.head      = nn.Sequential(nn.Linear(hidden, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(dropout*.4), nn.Linear(64, n_out))
    def _encode(self, x):
        x = self.entry(x)
        for b in self.blocks: x = b(x)
        return x
    def project(self, x): return self.projector(self._encode(x))
    def forward(self, x): return self.head(self._encode(x))

# ── CompactDomainFTT ─────────────────────────────────────────────
class DomainStem(nn.Module):
    def __init__(self, in_dim, token_dim, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, token_dim*2), nn.BatchNorm1d(token_dim*2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(token_dim*2, token_dim), nn.BatchNorm1d(token_dim))
    def forward(self, x): return self.net(x)

class CompactDomainFTT(nn.Module):
    def __init__(self, domain_indices, token_dim=64, n_heads=4, n_layers=2,
                 dropout=0.15, proj_dim=64, n_out=3):
        super().__init__()
        self.idx   = domain_indices
        self.stems = nn.ModuleDict({n: DomainStem(len(ids), token_dim, dropout)
                                    for n, ids in domain_indices.items()})
        self.cls_token   = nn.Parameter(torch.randn(1, 1, token_dim) * 0.02)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=token_dim, nhead=n_heads, dim_feedforward=token_dim*2,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        pool_dim = token_dim * 2
        self.projector = nn.Sequential(nn.Linear(pool_dim, pool_dim), nn.GELU(), nn.Linear(pool_dim, proj_dim))
        self.head = nn.Sequential(
            nn.Linear(pool_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout*.5),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout*.2), nn.Linear(64, n_out))
    def _encode(self, x):
        tokens = [self.stems[n](x[:, self.idx[n]]).unsqueeze(1) for n in self.stems]
        cls    = self.cls_token.expand(x.shape[0], -1, -1)
        seq    = torch.cat([cls] + tokens, dim=1)
        seq    = self.transformer(seq)
        return torch.cat([seq[:, 0], seq[:, 1:].mean(1)], dim=1)
    def project(self, x): return self.projector(self._encode(x))
    def forward(self, x): return self.head(self._encode(x))

# ── DualStreamDomainNet ──────────────────────────────────────────
class DualStreamBranch(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.25):
        super().__init__()
        wide_d    = out_dim * 3
        self.wide = nn.Sequential(
            nn.Linear(in_dim, wide_d), nn.BatchNorm1d(wide_d), nn.GELU(),
            nn.Dropout(dropout*.5), nn.Linear(wide_d, out_dim), nn.BatchNorm1d(out_dim))
        self.entry = nn.Linear(in_dim, out_dim)
        self.res1  = nn.Sequential(nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim))
        self.res2  = nn.Sequential(nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim), nn.GELU(),
                                   nn.Dropout(dropout*.6), nn.Linear(out_dim, out_dim), nn.BatchNorm1d(out_dim))
        self.act      = nn.GELU()
        self.gate_in  = nn.Sequential(nn.Linear(in_dim, in_dim), nn.Sigmoid())
        self.fuse     = nn.Sequential(nn.Linear(out_dim*2, out_dim), nn.Sigmoid())
    def forward(self, x):
        xg = x * self.gate_in(x); hw = self.wide(xg)
        hd = self.act(self.entry(xg)); hd = self.act(hd + self.res1(hd)); hd = self.act(hd + self.res2(hd))
        alpha = self.fuse(torch.cat([hw, hd], dim=1))
        return alpha * hw + (1 - alpha) * hd

class DualStreamDomainNet(nn.Module):
    def __init__(self, domain_indices, branch_dim=64, n_heads=4, dropout=0.25, proj_dim=64, n_out=3):
        super().__init__()
        self.idx      = domain_indices
        self.branches = nn.ModuleDict({n: DualStreamBranch(len(ids), branch_dim, dropout)
                                       for n, ids in domain_indices.items()})
        assert branch_dim % n_heads == 0
        self.norm1 = nn.LayerNorm(branch_dim)
        self.attn  = nn.MultiheadAttention(branch_dim, n_heads, dropout=0.1, batch_first=True)
        self.norm2 = nn.LayerNorm(branch_dim)
        self.ff    = nn.Sequential(nn.Linear(branch_dim, branch_dim*2), nn.GELU(),
                                   nn.Dropout(0.1), nn.Linear(branch_dim*2, branch_dim))
        pool_dim   = branch_dim * 2
        self.projector = nn.Sequential(nn.Linear(pool_dim, pool_dim), nn.GELU(), nn.Linear(pool_dim, proj_dim))
        self.head = nn.Sequential(nn.Linear(pool_dim, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout*.5),
                                  nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout*.2), nn.Linear(64, n_out))
    def _pool(self, x):
        outs = [self.branches[n](x[:, self.idx[n]]) for n in self.branches]
        h    = torch.stack(outs, dim=1)
        h    = h + self.attn(self.norm1(h), self.norm1(h), self.norm1(h))[0]
        h    = h + self.ff(self.norm2(h))
        return torch.cat([h.mean(1), h.max(1).values], dim=1)
    def project(self, x): return self.projector(self._pool(x))
    def forward(self, x): return self.head(self._pool(x))

print("All neural architectures defined ✓")


All neural architectures defined ✓


## 8. Train All Neural Models (Both Streams)

In [10]:
# Hyperparameters (best from grid search in experimenting notebooks)
HP = {
    'GRNNet':              {'hidden': 256, 'n_blocks': 5, 'dropout': 0.25, 'lr': 3e-4, 'sc_alpha': 0.3, 'sc_temp': 0.1, 'focal_gamma': 2.0},
    'DeepResNet':          {'hidden': 256, 'n_blocks': 6, 'dropout': 0.25, 'lr': 3e-4, 'sc_alpha': 0.3, 'sc_temp': 0.1, 'focal_gamma': 2.0},
    'CompactDomainFTT':    {'token_dim': 64, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.15, 'lr': 2e-4, 'sc_alpha': 0.3, 'sc_temp': 0.1, 'focal_gamma': 2.0},
    'DualStreamDomainNet': {'branch_dim': 64, 'n_heads': 4, 'dropout': 0.25, 'lr': 2e-4, 'sc_alpha': 0.3, 'sc_temp': 0.1, 'focal_gamma': 2.0},
}

trained_models = {'full': {}, 'alg': {}}
val_scores     = {'full': {}, 'alg': {}}
all_results    = {'full': [], 'alg': []}


def train_stream(stream_tag, X_tr, y_tr, Xtr, ytr, Xval, yval, X_te, y_te,
                 n_feat, idx, cw_tensor):
    hp = HP['GRNNet']
    print(f"\n--- GRNNet [{stream_tag}] ---")
    m = GRNNet(n_feat, hidden=hp['hidden'], n_blocks=hp['n_blocks'], dropout=hp['dropout']).to(device)
    vf1, _ = fit(m, Xtr, ytr, Xval, yval, lr=hp['lr'], sc_alpha=hp['sc_alpha'],
                 sc_temp=hp['sc_temp'], focal_gamma=hp['focal_gamma'], class_weights_tensor=cw_tensor)
    r = score_model(m, X_te, y_te, 'GRNNet', stream_tag)
    trained_models[stream_tag]['GRNNet'] = m; val_scores[stream_tag]['GRNNet'] = vf1
    all_results[stream_tag].append(r)

    hp = HP['DeepResNet']
    print(f"\n--- DeepResNet-SE [{stream_tag}] ---")
    m = DeepResNet(n_feat, hidden=hp['hidden'], n_blocks=hp['n_blocks'], dropout=hp['dropout']).to(device)
    vf1, _ = fit(m, Xtr, ytr, Xval, yval, lr=hp['lr'], sc_alpha=hp['sc_alpha'],
                 sc_temp=hp['sc_temp'], focal_gamma=hp['focal_gamma'], class_weights_tensor=cw_tensor)
    r = score_model(m, X_te, y_te, 'DeepResNet-SE', stream_tag)
    trained_models[stream_tag]['DeepResNet-SE'] = m; val_scores[stream_tag]['DeepResNet-SE'] = vf1
    all_results[stream_tag].append(r)

    hp = HP['CompactDomainFTT']
    print(f"\n--- CompactDomainFTT [{stream_tag}] ---")
    m = CompactDomainFTT(idx, token_dim=hp['token_dim'], n_heads=hp['n_heads'], n_layers=hp['n_layers'], dropout=hp['dropout']).to(device)
    vf1, _ = fit(m, Xtr, ytr, Xval, yval, lr=hp['lr'], sc_alpha=hp['sc_alpha'],
                 sc_temp=hp['sc_temp'], focal_gamma=hp['focal_gamma'], class_weights_tensor=cw_tensor)
    r = score_model(m, X_te, y_te, 'CompactDomainFTT', stream_tag)
    trained_models[stream_tag]['CompactDomainFTT'] = m; val_scores[stream_tag]['CompactDomainFTT'] = vf1
    all_results[stream_tag].append(r)

    hp = HP['DualStreamDomainNet']
    print(f"\n--- DualStreamDomainNet [{stream_tag}] ---")
    m = DualStreamDomainNet(idx, branch_dim=hp['branch_dim'], n_heads=hp['n_heads'], dropout=hp['dropout']).to(device)
    vf1, _ = fit(m, Xtr, ytr, Xval, yval, lr=hp['lr'], sc_alpha=hp['sc_alpha'],
                 sc_temp=hp['sc_temp'], focal_gamma=hp['focal_gamma'], class_weights_tensor=cw_tensor)
    r = score_model(m, X_te, y_te, 'DualStreamDomainNet', stream_tag)
    trained_models[stream_tag]['DualStreamDomainNet'] = m; val_scores[stream_tag]['DualStreamDomainNet'] = vf1
    all_results[stream_tag].append(r)

print("Training FULL stream...")
train_stream('full', X_tr_full, y_tr_full, Xtr_f, ytr_f, Xval_f, yval_f,
             X_te_full, y_te_full, N_FEAT_FULL, IDX_FULL, cw_full)

print("\nTraining ALG stream...")
train_stream('alg', X_tr_alg, y_tr_alg, Xtr_a, ytr_a, Xval_a, yval_a,
             X_te_alg, y_te_alg, N_FEAT_ALG, IDX_ALG, cw_alg)


Training FULL stream...

--- GRNNet [full] ---
  [full] GRNNet: Acc=0.8966  F1=0.8982  AUC=0.9686
              precision    recall  f1-score   support

          CN       0.92      0.91      0.91      1643
         MCI       0.89      0.92      0.91       826
    Dementia       0.88      0.87      0.88      1671

    accuracy                           0.90      4140
   macro avg       0.90      0.90      0.90      4140
weighted avg       0.90      0.90      0.90      4140


--- DeepResNet-SE [full] ---
  [full] DeepResNet-SE: Acc=0.8894  F1=0.8910  AUC=0.9700
              precision    recall  f1-score   support

          CN       0.91      0.91      0.91      1643
         MCI       0.88      0.91      0.90       826
    Dementia       0.87      0.86      0.86      1671

    accuracy                           0.89      4140
   macro avg       0.89      0.89      0.89      4140
weighted avg       0.89      0.89      0.89      4140


--- CompactDomainFTT [full] ---
  [full] CompactDom

## 9. Train LightGBM (Both Streams)

In [11]:
lgbm_models = {}

def train_lightgbm(stream_tag, X_tr, y_tr, X_te, y_te):
    if not HAS_LGBM:
        print("LightGBM not available — skipping"); return None
    print(f"\nTraining LightGBM [{stream_tag}]...")
    model = LGBMClassifier(
        n_estimators=600, num_leaves=63, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        class_weight='balanced', objective='multiclass', num_class=N_CLASSES,
        n_jobs=-1, random_state=SEED, verbose=-1
    )
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)
    f1  = f1_score(y_te, preds, average='macro', zero_division=0)
    acc = accuracy_score(y_te, preds)
    try:
        auc = roc_auc_score(np.eye(N_CLASSES)[y_te], probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    print(f"  [{stream_tag}] LightGBM: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(y_te, preds, target_names=CLASS_NAMES, zero_division=0))
    lgbm_models[stream_tag] = model
    all_results[stream_tag].append({'name': 'LightGBM', 'stream': stream_tag,
                                    'accuracy': acc, 'f1_macro': f1, 'auc_ovr': auc})
    return model

lgbm_models['full'] = train_lightgbm('full', X_tr_full, y_tr_full, X_te_full, y_te_full)
lgbm_models['alg']  = train_lightgbm('alg',  X_tr_alg,  y_tr_alg,  X_te_alg,  y_te_alg)



Training LightGBM [full]...
  [full] LightGBM: Acc=0.9316  F1=0.9314  AUC=0.9863
              precision    recall  f1-score   support

          CN       0.94      0.95      0.95      1643
         MCI       0.94      0.92      0.93       826
    Dementia       0.92      0.92      0.92      1671

    accuracy                           0.93      4140
   macro avg       0.93      0.93      0.93      4140
weighted avg       0.93      0.93      0.93      4140


Training LightGBM [alg]...
  [alg] LightGBM: Acc=0.9304  F1=0.9302  AUC=0.9865
              precision    recall  f1-score   support

          CN       0.94      0.95      0.95      1643
         MCI       0.94      0.92      0.93       826
    Dementia       0.92      0.92      0.92      1671

    accuracy                           0.93      4140
   macro avg       0.93      0.93      0.93      4140
weighted avg       0.93      0.93      0.93      4140



## 10. Build Soft-Vote Ensemble (Top-3 Neural Models)

In [12]:
ensemble_weights = {}  # {stream_tag: {model_name: weight}}

def build_ensemble(stream_tag, X_te, y_te):
    vs = val_scores[stream_tag]; tm = trained_models[stream_tag]
    if len(tm) < 2:
        print(f"[{stream_tag}] Not enough models for ensemble."); return
    top_n   = min(3, len(vs))
    top_nms = sorted(vs, key=vs.get, reverse=True)[:top_n]
    w       = np.array([vs[n] for n in top_nms]); w /= w.sum()
    print(f"[{stream_tag}] Ensemble members: {list(zip(top_nms, w.round(3)))}")
    ensemble_weights[stream_tag] = dict(zip(top_nms, w.tolist()))

    ens_probs = sum(w_i * get_probs(tm[n], X_te) for n, w_i in zip(top_nms, w))
    preds     = ens_probs.argmax(1)
    f1  = f1_score(y_te, preds, average='macro', zero_division=0)
    acc = accuracy_score(y_te, preds)
    try:
        auc = roc_auc_score(np.eye(N_CLASSES)[y_te], ens_probs, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    print(f"  [{stream_tag}] Ensemble: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(y_te, preds, target_names=CLASS_NAMES, zero_division=0))
    all_results[stream_tag].append({'name': f'Ensemble(top-{top_n})', 'stream': stream_tag,
                                    'accuracy': acc, 'f1_macro': f1, 'auc_ovr': auc})

build_ensemble('full', X_te_full, y_te_full)
build_ensemble('alg',  X_te_alg,  y_te_alg)


[full] Ensemble members: [('DeepResNet-SE', np.float64(0.336)), ('GRNNet', np.float64(0.333)), ('DualStreamDomainNet', np.float64(0.331))]
  [full] Ensemble: Acc=0.9034  F1=0.9039  AUC=0.9800
              precision    recall  f1-score   support

          CN       0.92      0.93      0.92      1643
         MCI       0.89      0.92      0.91       826
    Dementia       0.89      0.87      0.88      1671

    accuracy                           0.90      4140
   macro avg       0.90      0.91      0.90      4140
weighted avg       0.90      0.90      0.90      4140

[alg] Ensemble members: [('GRNNet', np.float64(0.335)), ('DeepResNet-SE', np.float64(0.334)), ('DualStreamDomainNet', np.float64(0.331))]
  [alg] Ensemble: Acc=0.9056  F1=0.9060  AUC=0.9807
              precision    recall  f1-score   support

          CN       0.92      0.93      0.93      1643
         MCI       0.90      0.92      0.91       826
    Dementia       0.90      0.87      0.88      1671

    accuracy       

## 11. Results Summary

In [13]:
import pandas as pd
for stream_tag in ['full', 'alg']:
    df = pd.DataFrame(all_results[stream_tag]).sort_values('f1_macro', ascending=False).reset_index(drop=True)
    print(f"\n{'='*60}")
    print(f"  {stream_tag.upper()} stream results")
    print('='*60)
    print(df[['name','accuracy','f1_macro','auc_ovr']].round(4).to_string(index=False))



  FULL stream results
               name  accuracy  f1_macro  auc_ovr
           LightGBM    0.9316    0.9314   0.9863
    Ensemble(top-3)    0.9034    0.9039   0.9800
             GRNNet    0.8966    0.8982   0.9686
      DeepResNet-SE    0.8894    0.8910   0.9700
DualStreamDomainNet    0.8819    0.8811   0.9720
   CompactDomainFTT    0.8316    0.8357   0.9489

  ALG stream results
               name  accuracy  f1_macro  auc_ovr
           LightGBM    0.9304    0.9302   0.9865
    Ensemble(top-3)    0.9056    0.9060   0.9807
      DeepResNet-SE    0.8937    0.8951   0.9703
             GRNNet    0.8884    0.8891   0.9663
DualStreamDomainNet    0.8841    0.8835   0.9733
   CompactDomainFTT    0.8396    0.8421   0.9511


## 12. Save Models & Artifacts

In [14]:
def save_stream(stream_tag, feature_cols, scaler, le_lbl, glb_med, str_le_map):
    out_dir = f'{SAVE_ROOT}/{stream_tag}'
    os.makedirs(out_dir, exist_ok=True)

    # Save LightGBM
    if lgbm_models.get(stream_tag):
        joblib.dump(lgbm_models[stream_tag], f'{out_dir}/lightgbm.pkl')
        print(f"  Saved LightGBM → {out_dir}/lightgbm.pkl")

    # Save ensemble: model weights + member weights
    ew = ensemble_weights.get(stream_tag, {})
    tm = trained_models[stream_tag]
    for name, model in tm.items():
        safe = name.lower().replace(' ', '_').replace('-', '_')
        torch.save(model.state_dict(), f'{out_dir}/{safe}_state.pt')
    joblib.dump(ew, f'{out_dir}/ensemble_weights.pkl')
    print(f"  Saved {len(tm)} neural model state dicts + ensemble weights")

    # Save preprocessing artifacts
    joblib.dump(scaler,      f'{out_dir}/scaler.pkl')
    joblib.dump(le_lbl,      f'{out_dir}/label_encoder.pkl')
    joblib.dump(feature_cols, f'{out_dir}/feature_cols.pkl')
    joblib.dump(glb_med,     f'{out_dir}/global_medians.pkl')
    joblib.dump(str_le_map,  f'{out_dir}/string_encoders.pkl')

    # Save results
    pd.DataFrame(all_results[stream_tag]).sort_values('f1_macro', ascending=False).to_csv(
        f'{out_dir}/results.csv', index=False)
    print(f"  Saved preprocessing artifacts → {out_dir}/")

save_stream('full', AF_FULL, scaler_full, le_full, glb_med_full, str_le_full)
save_stream('alg',  AF_ALG,  scaler_alg,  le_alg,  glb_med_alg,  str_le_alg)
print("\nAll artifacts saved!")


  Saved LightGBM → ./diagnosis_models/full/lightgbm.pkl
  Saved 4 neural model state dicts + ensemble weights
  Saved preprocessing artifacts → ./diagnosis_models/full/
  Saved LightGBM → ./diagnosis_models/alg/lightgbm.pkl
  Saved 4 neural model state dicts + ensemble weights
  Saved preprocessing artifacts → ./diagnosis_models/alg/

All artifacts saved!


## 13. Inference Functions — Use After Loading Saved Models

In [15]:
# ─────────────────────────────────────────────────────────────────
# LOADING UTILITIES (run these in a fresh session to do inference)
# ─────────────────────────────────────────────────────────────────

def load_diagnosis_artifacts(stream_tag, save_root='./diagnosis_models'):
    """Load all saved artifacts for one stream. Returns a dict."""
    out_dir = f'{save_root}/{stream_tag}'
    arts = {}
    arts['scaler']         = joblib.load(f'{out_dir}/scaler.pkl')
    arts['label_encoder']  = joblib.load(f'{out_dir}/label_encoder.pkl')
    arts['feature_cols']   = joblib.load(f'{out_dir}/feature_cols.pkl')
    arts['global_medians'] = joblib.load(f'{out_dir}/global_medians.pkl')
    arts['string_encoders']= joblib.load(f'{out_dir}/string_encoders.pkl')
    arts['ensemble_weights']= joblib.load(f'{out_dir}/ensemble_weights.pkl')
    lgbm_path = f'{out_dir}/lightgbm.pkl'
    if os.path.exists(lgbm_path):
        arts['lightgbm'] = joblib.load(lgbm_path)
    return arts


def preprocess_instance(instance_dict, arts):
    """
    Preprocess a single visit dict for inference.
    instance_dict: {feature_name: value, ...}  (raw, unscaled)
    arts: output of load_diagnosis_artifacts()
    Returns: scaled numpy array (1, n_features)
    """
    feat_cols    = arts['feature_cols']
    glb_med      = arts['global_medians']
    str_encoders = arts['string_encoders']
    scaler       = arts['scaler']

    row = {}
    for col in feat_cols:
        if col.endswith('_missing'):
            base = col[:-len('_missing')]
            row[col] = 1.0 if (base not in instance_dict or pd.isna(instance_dict.get(base))) else 0.0
        elif col in str_encoders:
            val = instance_dict.get(col, 'not_recorded')
            le  = str_encoders[col]
            val = str(val) if not pd.isna(val) else 'not_recorded'
            if val in le.classes_:
                row[col] = float(le.transform([val])[0])
            else:
                row[col] = float(le.transform([le.classes_[0]])[0])  # fallback to first class
        else:
            val = instance_dict.get(col, np.nan)
            if pd.isna(val):
                val = glb_med.get(col, 0.0)
            row[col] = float(val)

    X_raw = np.array([[row[c] for c in feat_cols]], dtype=np.float32)
    return scaler.transform(X_raw).astype(np.float32)


def predict_lightgbm(instance_dict, arts):
    """Predict diagnosis using LightGBM for a single visit."""
    if 'lightgbm' not in arts:
        raise RuntimeError("LightGBM not found in artifacts — check save path")
    X = preprocess_instance(instance_dict, arts)
    model = arts['lightgbm']
    probs = model.predict_proba(X)[0]
    pred  = int(np.argmax(probs))
    le    = arts['label_encoder']
    label = le.inverse_transform([pred])[0]
    return {
        'prediction': label,
        'class_index': pred,
        'probabilities': {le.inverse_transform([i])[0]: float(p) for i, p in enumerate(probs)},
        'class_names': CLASS_NAMES
    }


def predict_ensemble(instance_dict, arts, neural_models_dict, device_inf=None):
    """
    Predict using the saved soft-vote ensemble of neural models.
    neural_models_dict: {model_name: loaded_pytorch_model}  (already in eval mode)
    """
    if device_inf is None:
        device_inf = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X = preprocess_instance(instance_dict, arts)
    Xt = torch.tensor(X, dtype=torch.float32).to(device_inf)
    ew = arts['ensemble_weights']  # {name: weight}
    ens_probs = np.zeros(N_CLASSES, dtype=np.float32)
    for name, weight in ew.items():
        model = neural_models_dict[name]
        model.eval()
        with torch.no_grad():
            p = F.softmax(model(Xt), dim=1).cpu().numpy()[0]
        ens_probs += weight * p
    pred  = int(np.argmax(ens_probs))
    le    = arts['label_encoder']
    label = le.inverse_transform([pred])[0]
    return {
        'prediction': label,
        'class_index': pred,
        'probabilities': {le.inverse_transform([i])[0]: float(p) for i, p in enumerate(ens_probs)},
        'ensemble_members': list(ew.keys()),
        'class_names': CLASS_NAMES
    }

print("Inference functions ready.")


Inference functions ready.


## 14. Example Inference (Demo)

In [16]:
# ── Demo: predict a single visit from test set ───────────────────────────
arts_alg = load_diagnosis_artifacts('alg')

# Build a demo instance from the first test row
feat_cols = arts_alg['feature_cols']
row_dict  = {c: v for c, v in zip(feat_cols, X_te_alg[0])}  # already scaled — for demo only

# For real new data you'd pass raw unscaled values:
# row_dict = {'MMSCORE': 24, 'MOCA': 22, 'age': 72, 'PTGENDER': 'Female', ...}

result_lgbm = predict_lightgbm(row_dict, arts_alg)
print("LightGBM prediction:", result_lgbm)

# For ensemble inference, reload the neural models first:
# arts = load_diagnosis_artifacts('alg')
# neural_models = {
#     'GRNNet': GRNNet(N_FEAT_ALG, ...).to(device); load_state_dict(torch.load('diagnosis_models/alg/grnnet_state.pt'))
#     ...
# }
# result_ens = predict_ensemble(row_dict, arts, neural_models)
print("\nNote: for full ensemble inference, load neural model state dicts and pass them to predict_ensemble().")


LightGBM prediction: {'prediction': np.int64(1), 'class_index': 1, 'probabilities': {np.int64(0): 0.0010844129368434434, np.int64(1): 0.6143525555696527, np.int64(2): 0.3845630314935037}, 'class_names': ['CN', 'MCI', 'Dementia']}

Note: for full ensemble inference, load neural model state dicts and pass them to predict_ensemble().
